# Gene-panel preprocessing — one configured run

One pass of the whole preprocessing pipeline for **one explicit choice** of
reference data, embedding method, similarity score and parameters.
Everything is set in the **Parameters** cell below; the rest of the notebook is
thin and only calls `panelprep` functions.

**What it produces**

| # | output | where |
|---|--------|-------|
| 1 | candidate datasets scored against the measured panel, red/blue heatmap, comparison report | `<outdir>/score_results.csv`, `<outdir>/figures/score_heatmap*.png`, `<outdir>/comparison_report.md` |
| 2 | gene embeddings `G x d` from the expression matrices, separate **and** joint | `<outdir>/store/expression/*.parquet` |
| 3 | gene embeddings from protein sequence (ESM-2), PPI (STRING), literature (BioGPT), regulation (KEGG) | `<outdir>/store/prior/*.parquet` — **loaded, never retrained here** |
| 4 | pathway / ligand / receptor / TF annotations | `<outdir>/gene_annotation.csv` |
| — | every figure, plus the inventory of what exists and its shape | `<outdir>/figures/`, `<outdir>/embedding_inventory.csv` |

**Cost.** With the store already populated this notebook is seconds to a few
minutes. The protein-language-model and graph embeddings are *not* built here —
they depend only on the gene list, cost 35–50 minutes each, and are produced
once by:

```bash
python scripts/build_embedding_store.py --config configs/real_pilot.yaml
```

If one is requested and missing, the run reports it as `blocked` and continues.

In [ ]:
# Setup. Run from the repository root (or the Morpholab repo's preprocessing/ folder).
import os, sys, warnings
from pathlib import Path

if Path('panelprep').is_dir():
    ROOT = Path('.').resolve()
else:                                  # launched from notebooks/
    ROOT = Path('..').resolve()
    os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 50)
warnings.filterwarnings('ignore', category=FutureWarning)

from panelprep.config import Config
from panelprep.run import run_analysis, CACHED_ONLY
from panelprep.embed import EMBEDDERS
from panelprep.embed.store import EmbeddingStore
from panelprep.scores import SCORES
from panelprep.io.catalog import catalog_table

# Registering the Morpholab adapter adds its score (corr_mae) and its gene
# representation (factor_analysis) to the registries. Optional: comment out the
# import and both simply disappear from the menus below.
import panelprep.integrations.morpholab as morpholab

print(f'root            : {ROOT}')
print(f'scores available: {", ".join(SCORES.names())}')
print(f'embedders       : {", ".join(EMBEDDERS.names())}')

## Parameters

**This is the only cell to edit.** Each entry names one choice; the tables
printed above list the legal values.

In [ ]:
# ============================ PARAMETERS ===================================
# --- 1. the measured input -------------------------------------------------
PANEL = ['T', 'Krt8', 'Mest', 'Dnmt3b']      # the measured gene subset S
QUERY_FILE = 'GSM9046243_Embryo_E7.5_stereo_rep1.h5ad'   # under data/<accession>/
QUERY_MAX_OBS = 4000                          # cells kept from the query
DESCRIPTOR = (
    'Spatial transcriptomics of a whole mouse embryo at embryonic day 7.5 (E7.5) '
    'during gastrulation and early organogenesis. Cells retain tissue coordinates. '
    'We study germ layer patterning and the emergence of mesoderm and endoderm '
    'progenitors.')

# --- 2. reference data (the candidate set D) -------------------------------
# Names from catalog_table(). Drop entries to make the run smaller.
CANDIDATES = [
    'seu3d_E7.5_rep2',        # spatial, stage-matched
    'seu3d_E7.75_rep2',       # spatial, +0.25 day
    'seu3d_E8.0_rep1',        # spatial, +0.5 day
    'mittnenzweig_wt_E7.5',   # single cell, stage-matched
    'mittnenzweig_wt_E6.5',   # single cell, deliberately mismatched (control)
    'mittnenzweig_foxc_dko',  # causal (Foxc1/Foxc2 double knockout)
]

# --- 3. similarity scores --------------------------------------------------
# Any subset of SCORES.names(). Adding one here updates the heatmap and the
# comparison report with no other edit.
SCORE_NAMES = [
    'avg_js',                      # symmetric per-gene distribution divergence
    'corr_frobenius',              # RMS panel-correlation difference (Spearman)
    'corr_cosine',                 # pattern of co-expression, scale-free
    'corr_mae',                    # the Morpholab select_dataset score (Pearson, MAE)
    'ot_cluster_ari',              # optimal transport between cell clusters
    'spatial_autocorr_frobenius',  # section-wise Moran's I agreement (spatial only)
]

# --- 4. embedding methods --------------------------------------------------
# Expression embedders are computed on a cache miss; CACHED_ONLY sources are
# loaded from the store and never retrained here.
EMBEDDERS_SEPARATE = ['pca_baseline', 'ae_separate', 'vae_separate']  # one per dataset
EMBEDDERS_JOINT    = ['joint_ae', 'dcca', 'contrastive_ae']           # one, integrated
EMBEDDERS_PRIOR    = ['protein_sequence', 'ppi_network',               # loaded only
                      'literature_text', 'regulatory_kegg', 'coexpression_atlas']

# --- 5. other parameters ---------------------------------------------------
EMBEDDING_DIM = 32          # d, the width of every expression embedding
MAX_GENES     = 2000        # size of the gene universe (panel genes always pinned)
GENE_UNIVERSE = 'intersection'   # 'union' or 'intersection' across datasets
SEED          = 0
PROJECTION    = 'umap'      # 'umap' | 'pca' | 'tsne'  (figures only)
COLOR_BY      = 'top_pathway'    # annotation column used for colour
# Gene names written on the embedding plots. The panel genes are always starred
# and labelled; these name additional genes.
LABEL_GENES   = ['Sox2', 'Pou5f1', 'Foxa2', 'Hand1', 'Epcam', 'Postn']  # or None
N_GENE_LABELS = 0           # >0 instead names the N most peripheral genes
OUTDIR        = 'outputs/notebook_run'
STORE_DIR     = 'outputs/real_pilot/store'   # where the cached embeddings live
RECOMPUTE     = False       # True = rebuild expression embeddings from scratch
# ===========================================================================

config = Config.from_yaml('configs/real_pilot.yaml').replace(
    run_name='notebook_run', outdir=OUTDIR, seed=SEED,
    candidates=CANDIDATES, scores=SCORE_NAMES,
    embedders=EMBEDDERS_SEPARATE + EMBEDDERS_JOINT + EMBEDDERS_PRIOR,
    embedding_dim=EMBEDDING_DIM, max_genes=MAX_GENES, gene_universe=GENE_UNIVERSE)

store = EmbeddingStore(STORE_DIR)
print(store)
print(f'\ncache-only sources present : '
      f'{[s for s in EMBEDDERS_PRIOR if store.has(s)]}')
print(f'cache-only sources missing: '
      f'{[s for s in EMBEDDERS_PRIOR if not store.has(s)]}')

### The catalog, the scores and the embedders on offer

All three menus are generated from the registries, so anything you add to the
package shows up here automatically.

In [ ]:
display(Markdown('**Candidate datasets in the catalog**'))
display(catalog_table())
# Both meta tables are indexed by the registry name.
display(Markdown('**Scores** (orientation = which direction means "agrees")'))
display(SCORES.meta_table()[['orientation', 'applies_to', 'units_sensitive', 'description']])
display(Markdown('**Embedders** (`mode`: separate = one per dataset, joint = one integrated)'))
display(EMBEDDERS.meta_table()[['mode', 'category', 'description']])

## 1. The measured input

`PanelQuery` holds the `N x |S|` expression matrix, the `N x 2` coordinates and
the free-text descriptor. Note it carries **only the panel columns** — that is
the premise of the problem.

In [ ]:
from panelprep.io.loaders import query_from_seu3d

query = query_from_seu3d(QUERY_FILE, PANEL, DESCRIPTOR, max_obs=QUERY_MAX_OBS,
                         rng=np.random.default_rng(SEED), name='query')
print(f'expression {query.expression.shape}   coords {query.coords.shape}   units {query.units}')
print(f'detection rate per panel gene: '
      f'{(query.expression > 0).mean().round(3).to_dict()}')
query.expression.describe().T.round(3)

In [ ]:
# The panel, in space. Sanity check before anything else: if these look wrong,
# nothing downstream can be right.
fig, axes = plt.subplots(1, len(PANEL), figsize=(3.6 * len(PANEL), 3.4),
                         constrained_layout=True)
for ax, g in zip(np.atleast_1d(axes), PANEL):
    s = ax.scatter(query.coords[:, 0], query.coords[:, 1], c=query.expression[g],
                   s=3, cmap='magma', vmin=0)
    ax.set_title(f'{g}   detected in {float((query.expression[g] > 0).mean()):.1%}')
    ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(s, ax=ax, shrink=0.8, label='log1p CP10K')
fig.suptitle('Measured panel on the query embryo (sagittal projection)')
fig

## 2. Run everything

One call. It loads the references, scores them, fixes the gene universe with the
panel pinned in, obtains every embedding (building the cheap ones, loading the
expensive ones), annotates the genes, renders the figures and writes the report.

In [ ]:
result = run_analysis(
    config,
    query=query,
    store=store,
    recompute=RECOMPUTE,
    projection=PROJECTION,
    group_figures_by='store_group',   # 'method' | 'category' | 'dataset' | None
    verbose=True,
)

if result['blocked']:
    display(Markdown('**Blocked** (reported, not silently skipped):'))
    for name, why in result['blocked'].items():
        print(f'  - {name}: {why}')

## 3. Which reference datasets match the panel?

Red = agrees most with the measured panel, blue = least. The companion
`score_heatmap_raw.png` colours by the raw numeric value instead, because a
*low* divergence and a *high* agreement are the same fact with opposite signs
and only one of them should be red.

In [ ]:
display(result['score_table'].round(4))
display(Image(result['figures']['score_heatmap_agreement']))
display(Image(result['figures']['score_heatmap_raw']))

In [ ]:
# The ranking behind the colours, and the per-score detail with coverage.
if result.get('ranking') is not None:
    display(result['ranking'].round(4))
display(result['scores'][['dataset', 'score', 'value', 'status',
                          'n_genes_used', 'n_genes_requested']].round(5))

## 4. What embeddings exist, what shape, and where they are saved

Every embedder returns a `GeneEmbedding` whose `.matrix` is a
`|G| x d` DataFrame — **one row per gene, indexed by gene symbol**, `float32`.
A `separate` embedder returns one matrix *per dataset*; a `joint` embedder
returns one matrix built from all of them at once. Prior sources return their
own native width (ESM-2 480, BioGPT 1024, the graph sources `dim`).

In [ ]:
inv = result['inventory']
display(inv[['name', 'group', 'method', 'category', 'shape',
             'n_nan_genes', 'size_kb', 'path']])
print(f"{len(inv)} embeddings, {inv['size_kb'].sum() / 1024:.1f} MB total")
print(f"store root: {result['store_root']}")

In [ ]:
# Read one back off disk and confirm the contract.
name = sorted(result['embeddings'])[0]
emb = store.get(name)
print(f'{name}: {type(emb).__name__}, matrix {emb.matrix.shape}, '
      f'dtype {emb.matrix.dtypes.iloc[0]}, category {emb.category}')
print(f'panel genes present: {[g for g in PANEL if g in emb.matrix.index]}')
display(emb.matrix.loc[[g for g in PANEL if g in emb.matrix.index]].iloc[:, :6].round(3))

## 5. Gene annotations

Pathway membership and ligand / receptor / transcription-factor roles — the
labels the embedding figures are coloured by.

In [ ]:
ann = result.get('annotation')
if ann is not None:
    t = ann.table
    print(f'{len(t)} genes annotated; '
          f'{int(t["is_ligand"].sum())} ligands, {int(t["is_receptor"].sum())} receptors, '
          f'{int(t["is_tf"].sum())} transcription factors')
    display(Markdown('**The measured panel:**'))
    display(t.loc[[g for g in PANEL if g in t.index],
                  ['top_pathway', 'is_ligand', 'is_receptor', 'is_tf']])
    display(Markdown('**Most common pathways in the universe:**'))
    display(t['top_pathway'].value_counts().head(12).to_frame('n_genes'))

## 6. Every figure

`visualize_all` renders the whole set: one panel figure per group of comparable
embeddings, the two method-by-method agreement matrices, the all-pairs grid and
one focused pair with arrows on the genes that move most. The measured panel
genes are drawn as large labelled stars.

In [ ]:
manifest = result['figure_manifest']
display(manifest[['figure', 'kind', 'group', 'n_embeddings']])
for _, row in manifest.iterrows():
    display(Markdown(f"**{row['figure']}** — {row['kind']}, {row['group']}"))
    display(Image(row['path']))

In [ ]:
# The numbers behind the panel figures: silhouette and kNN label agreement
# against a label-permutation null. A p near 1 means "indistinguishable from
# chance" -- i.e. that embedding does not recover pathway structure.
q = pd.read_csv(manifest['quality_csv'].iloc[0])
display(q[['embedding', 'silhouette', 'silhouette_p',
           'knn_agreement', 'knn_p', 'n_genes_scored']].round(4))

## 7. The prior embeddings, loaded not retrained

These four come from outside the expression data entirely: protein sequence,
the interaction graph, the literature and the regulatory graph. They are read
from the store — building them costs ~35–50 minutes each and depends only on
the gene symbol list, so it happens once in
`scripts/build_embedding_store.py`.

Coverage is partial and **uncovered genes are NaN rows, never zeros** — a zero
vector is a position in the space and would cluster every uncovered gene
together.

In [ ]:
prior_names = store.names(group='prior')
priors = store.load_all(prior_names)
rows = []
for n, e in priors.items():
    cov = 1.0 - e.matrix.isna().all(axis=1).mean()
    rows.append({'source': n, 'category': e.category,
                 'shape': f'{e.matrix.shape[0]} x {e.matrix.shape[1]}',
                 'coverage': round(float(cov), 4),
                 'model_or_db': str(e.provenance.get('model_or_db_version',
                                     e.provenance.get('modality', '')))[:70],
                 'status': e.provenance.get('status', ''),
                 'path': e.provenance.get('loaded_from', '')})
display(pd.DataFrame(rows))

In [ ]:
# Visualise the priors on the genes they actually cover. Restricting to the
# run's gene universe keeps them comparable with the expression embeddings.
from panelprep.viz.gallery import visualize_all

prior_figs = visualize_all(
    store, result.get('annotation'),
    outdir=Path(OUTDIR) / 'figures_priors',
    names=prior_names, genes=result['genes'],
    group_by=None, method=PROJECTION, color_by=COLOR_BY,
    highlight=PANEL, seed=SEED, verbose=True)
for _, row in prior_figs.iterrows():
    display(Markdown(f"**{row['figure']}** — {row['kind']}"))
    display(Image(row['path']))

## 8. Where the measured panel sits among the unmeasured genes

The point of embedding all `|G|` genes rather than only the measured `|S|` is to
locate the panel inside the transcriptome. This is the view that answers
"what else is near what we measured".

In [ ]:
from panelprep.viz import embedding_scatter

focus = 'dcca' if 'dcca' in result['embeddings'] else sorted(result['embeddings'])[0]
sc = embedding_scatter(
    result['embeddings'][focus], result.get('annotation'),
    color_by=COLOR_BY, method=PROJECTION, highlight=PANEL, highlight_size=320,
    label_genes=LABEL_GENES, n_gene_labels=N_GENE_LABELS,
    out_path=str(Path(OUTDIR) / 'figures' / f'panel_in_context_{focus}.png'),
    title=f'{focus}: the measured panel inside the gene universe', seed=SEED)
display(Image(str(Path(OUTDIR) / 'figures' / f'panel_in_context_{focus}.png')))
print(sc.quality if hasattr(sc, 'quality') else '')

## 9. The comparison report

Generated from the registries and the long-format score frame, so a newly
registered score or embedder appears in it with no edit anywhere.

In [ ]:
display(Markdown(Path(result['figures']['comparison_report']).read_text()))

## Adding a score, an embedder, or a reference dataset

**A score** — write the function in `panelprep/scores/custom.py`:

```python
@SCORES.register('my_score', orientation='lower_is_better',
                 applies_to=('single_cell', 'spatial'),
                 needs_coords=False, units_sensitive=True,
                 description='one line for the report')
def my_score(query, candidate, *, genes, rng, **params):
    ...
    return ScoreResult(...)
```

then add `'my_score'` to `SCORE_NAMES` above. The heatmap, its colour
direction and the comparison report all follow the registry.

**An embedder** — same pattern in `panelprep/embed/custom.py` with
`@EMBEDDERS.register(..., mode='separate'|'joint')`, returning a
`GeneEmbedding` whose matrix is `|G| x d` indexed by gene symbol. Add it to
`EMBEDDERS_SEPARATE` or `EMBEDDERS_JOINT`. Put it in
`panelprep.run.CACHED_ONLY` if it is expensive and depends only on the gene
list.

**A reference dataset** — add an entry to `panelprep/io/catalog.py` with a
loader in `panelprep/io/loaders.py`, or, for a dataset registered in the
Morpholab repo, use the adapter:

```python
cand = morpholab.load_morpholab_dataset('tome', timepoint='7.5', layer='logcounts')
```